# Packages import

In [1]:
import os
import re
import yaml
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

## Apollo Scraper

In [2]:
# 252671 --> group code
group_id = input("Enter group ID: ")
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=1"

In [3]:
with open("config.yaml", "r", encoding="utf-8") as yf:
    config = yaml.load(yf, Loader=yaml.SafeLoader)
username = config.get("credentials", {}).get("username") if config else None
password = config.get("credentials", {}).get("password") if config else None
auth = HTTPBasicAuth(username, password)

In [4]:
response = requests.get(url, auth=auth)
response.encoding = "UTF-8"
print(response.status_code)

200


In [5]:
page_dom = BeautifulSoup(response.text, "html.parser")

In [6]:
group = page_dom.select_one("div.grupa").get_text()
print(group)

ZICSS1-1212


In [7]:
classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(str(classes_tag))

classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

In [8]:
classes = classes.loc[classes["Typ"].isin(["ćwiczenia", "wykład", "egzamin"])]

In [9]:
classes[["Day", "Start time", "hyphen", "End time", "Duration"]] = classes["Dzień, godzina"].str.split(" ", expand=True)

In [10]:
classes["Duration"] = classes["Duration"].map(lambda x: x.split("(")[1].split("g")[0])

In [11]:
classes = classes.drop(["Dzień, godzina", "hyphen"], axis=1)

In [12]:
classes["Sala"] = classes["Sala"].str.replace(
    r"(lab\.).*",
    r"\1",
    regex=True
)

In [13]:
if not os.path.exists("./schedules"):
    os.mkdir("./schedules")

In [14]:
classes.to_csv(f"schedules/{group}.csv", encoding="UTF-8")